In [2]:
import pandas as pd
import numpy as np
import altair as alt
import requests as rqs

### Certificado HTTPS do IBGE

O servidor omite o intermedi?rio Sectigo Public Server Authentication CA DV R36.
A sess?o abaixo complementa a cadeia com `certs/SectigoPublicServerAuthenticationCADVR36.pem`
e mant?m a verifica??o do dom?nio, da validade e da cadeia at? uma raiz confi?vel do `certifi`.
O ajuste se aplica somente a `https://servicodados.ibge.gov.br/`.
Execute o notebook com a pasta do projeto como diret?rio de trabalho.

Fonte do intermedi?rio: http://crt.sectigo.com/SectigoPublicServerAuthenticationCADVR36.crt

SHA-256 do certificado DER: `8c54c334b66ba4e426772af4a3f9136c19a1aec729fdb28c535c07a5a4ef22e0`.
A cadeia foi validada em 14/10/2026. Se o IBGE trocar de emissor, revise o intermedi?rio.


In [ ]:
import ssl
from pathlib import Path
from requests.adapters import HTTPAdapter


class IBGETLSAdapter(HTTPAdapter):
    """Completa a cadeia do IBGE sem desativar a valida??o TLS."""

    def __init__(self, certificado_intermediario, **kwargs):
        self.ssl_context = ssl.create_default_context(cafile=rqs.certs.where())
        # O intermedi?rio deve encadear at? uma raiz confi?vel, sem cadeia parcial.
        self.ssl_context.verify_flags &= ~ssl.VERIFY_X509_PARTIAL_CHAIN
        self.ssl_context.load_verify_locations(cafile=str(certificado_intermediario))
        super().__init__(**kwargs)

    def init_poolmanager(self, connections, maxsize, block=False, **pool_kwargs):
        pool_kwargs['ssl_context'] = self.ssl_context
        return super().init_poolmanager(connections, maxsize, block=block, **pool_kwargs)

    def proxy_manager_for(self, proxy, **proxy_kwargs):
        proxy_kwargs['ssl_context'] = self.ssl_context
        return super().proxy_manager_for(proxy, **proxy_kwargs)


In [ ]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/1612/metadados"
certificado_intermediario = Path("certs/SectigoPublicServerAuthenticationCADVR36.pem")

with rqs.Session() as sessao:
    sessao.mount("https://servicodados.ibge.gov.br/", IBGETLSAdapter(certificado_intermediario))
    response = sessao.get(url, timeout=30)
    response.raise_for_status()
    metadados = response.json()

In [9]:
print("Tabela:", metadados["nome"])

for variavel in metadados["variaveis"]:
    print(
        variavel["id"],
        variavel["nome"],
        variavel["unidade"]
    )

Tabela: Área plantada, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias
109 Área plantada Hectares
1000109 Área plantada - percentual do total geral %
216 Área colhida Hectares
1000216 Área colhida - percentual do total geral %
214 Quantidade produzida Toneladas
112 Rendimento médio da produção Quilogramas por Hectare
215 Valor da produção Mil Cruzeiros [1974 a 1985, 1990 a 1992], Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025]
1000215 Valor da produção - percentual do total geral %
